In [9]:
import os
import pandas as pd
import numpy as np

In [11]:
df = pd.read_csv("../data/raw/2019-Oct.csv")

In [14]:
df.isna().sum()

event_time              0
event_type              0
product_id              0
category_id             0
category_code    13515609
brand             6117080
price                   0
user_id                 0
user_session            2
dtype: int64

In [16]:
(df.isna().sum() / len(df) * 100).round(2)

event_time        0.00
event_type        0.00
product_id        0.00
category_id       0.00
category_code    31.84
brand            14.41
price             0.00
user_id           0.00
user_session      0.00
dtype: float64

Missing-value audit: The dataset is highly complete for core behavioural fields. Missing values are limited to product metadata, with 31.84% missing in category_code and 14.41% missing in brand. No cleaning action was taken during the audit stage.

In [19]:
df.duplicated().sum()

30220

In [27]:
(df.duplicated().sum() / len(df) * 100).round(2)

0.07

Duplicate audit: 30,220 rows (0.0712% of the dataset) are exact duplicates across all columns. Although the percentage is small, these records will be investigated before removal because identical behavioural events may not necessarily represent erroneous duplicates.

In [32]:
duplicates = df[df.duplicated(keep=False)]

duplicates.head(20)

,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
3829,2019-10-01 02:25:40 UTC,view,1307116,2053013558920217191,computers.notebook,hp,463.07,548049635,83085f41-22c3-4f27-bc29-bfaaf460d902
3830,2019-10-01 02:25:40 UTC,view,1307116,2053013558920217191,computers.notebook,hp,463.07,548049635,83085f41-22c3-4f27-bc29-bfaaf460d902
3831,2019-10-01 02:25:40 UTC,view,1307116,2053013558920217191,computers.notebook,hp,463.07,548049635,83085f41-22c3-4f27-bc29-bfaaf460d902
3832,2019-10-01 02:25:40 UTC,view,1307116,2053013558920217191,computers.notebook,hp,463.07,548049635,83085f41-22c3-4f27-bc29-bfaaf460d902
8564,2019-10-01 02:34:48 UTC,view,17200828,2053013559792632471,furniture.living_room.sofa,NaN,926.41,555464369,8bf38319-436a-4637-8c84-6de4213f8a13
8566,2019-10-01 02:34:48 UTC,view,17200828,2053013559792632471,furniture.living_room.sofa,NaN,926.41,555464369,8bf38319-436a-4637-8c84-6de4213f8a13
15515,2019-10-01 02:46:53 UTC,cart,1004833,2053013555631882655,electronics.smartphone,samsung,174.76,537717675,97a54368-3848-44ac-a249-14f84aeb3435
15517,2019-10-01 02:46:53 UTC,cart,1004833,2053013555631882655,electronics.smartphone,samsung,174.76,537717675,97a54368-3848-44ac-a249-14f84aeb3435
18816,2019-10-01 02:52:09 UTC,cart,1002544,2053013555631882655,electronics.smartphone,apple,464.13,535597803,1664252c-e49b-4e59-ae87-7a0d3a5a574c
18817,2019-10-01 02:52:09 UTC,cart,1002544,2053013555631882655,electronics.smartphone,apple,464.13,535597803,1664252c-e49b-4e59-ae87-7a0d3a5a574c


In [34]:
duplicates["event_type"].value_counts()

event_type
cart        44067
view         4100
purchase      152
Name: count, dtype: int64

In [36]:
duplicates.shape

(48319, 9)

### Duplicate Audit

- **30,220 rows** were identified as additional exact duplicates, representing approximately **0.0712%** of the full dataset.
- When all rows belonging to duplicate groups were displayed using `keep=False`, **48,319 records** were involved.
- Most duplicate-looking records were associated with **cart events**:
  - Cart: 44,067
  - View: 4,100
  - Purchase: 152

#### Observation

The concentration of duplicates within cart events suggests that some duplicate-looking records may represent genuine repeated customer interactions rather than simple data-entry errors.

Therefore, duplicate rows will **not be removed automatically**. Their event timing, user, product, and session information should be investigated before a cleaning decision is made.

In [41]:
print("Zero prices:", (df["price"] == 0).sum())
print("Negative prices:", (df["price"] < 0).sum())
print("Prices above 2000:", (df["price"] > 2000).sum())

Zero prices: 68673
Negative prices: 0
Prices above 2000: 227000


In [43]:
print("Zero price %:", ((df["price"] == 0).sum() / len(df) * 100))
print("Negative price %:", ((df["price"] < 0).sum() / len(df) * 100))
print("Above 2000 %:", ((df["price"] > 2000).sum() / len(df) * 100))

Zero price %: 0.16177856203304294
Negative price %: 0.0
Above 2000 %: 0.5347623313602252


### Price Quality Audit

- **Zero-price records:** 68,673
- **Zero-price percentage:** approximately 0.162%
- **Negative-price records:** 0
- **Records priced above 2,000:** 227,000
- **Percentage above 2,000:** approximately 0.535%

#### Observations

- No negative prices were found, which suggests that the price field does not contain obvious negative-value errors.
- A small number of records have a price of zero. These records require further investigation because a zero price may represent a data-quality issue, promotional/free items, or incomplete product pricing.
- Prices above 2,000 account for only a small proportion of the dataset. These values should not be automatically classified as outliers because the dataset contains products from many categories, including potentially high-value items.
- High-price records should be inspected by category, product, and brand before any cleaning decision is made.

In [49]:
print("Missing product_id:", df["product_id"].isna().sum())
print("Missing category_id:", df["category_id"].isna().sum())
print("Missing user_id:", df["user_id"].isna().sum())
print("Missing user_session:", df["user_session"].isna().sum())

print("Product IDs <= 0:", (df["product_id"] <= 0).sum())
print("Category IDs <= 0:", (df["category_id"] <= 0).sum())
print("User IDs <= 0:", (df["user_id"] <= 0).sum())

Missing product_id: 0
Missing category_id: 0
Missing user_id: 0
Missing user_session: 2
Product IDs <= 0: 0
Category IDs <= 0: 0
User IDs <= 0: 0


### ID and Session Validation

- `product_id` contains **no missing values**.
- `category_id` contains **no missing values**.
- `user_id` contains **no missing values**.
- `user_session` contains **2 missing values**.
- No `product_id`, `category_id`, or `user_id` values are zero or negative.

#### Observations

- The main identifier fields are highly complete and contain no obvious invalid numeric values.
- Only **2 session records are missing**, which is negligible compared with more than 42 million total rows.
- The earlier missing-value percentage displayed `0.00%` for `user_session` because the missing proportion is too small to appear when rounded to two decimal places.
- These two records should be inspected during the cleaning stage rather than automatically removed.

In [52]:
df["event_type"].value_counts(dropna=False)

event_type
view        40779399
cart          926516
purchase      742849
Name: count, dtype: int64

In [54]:
df["event_type"].nunique()

3

### Event Type Validation

The dataset contains exactly **3 unique event types**:

| Event Type | Count |
|---|---:|
| View | 40,779,399 |
| Cart | 926,516 |
| Purchase | 742,849 |

#### Observations

- No missing `event_type` values were found.
- No unexpected or inconsistent event labels were identified.
- The three valid event types are `view`, `cart`, and `purchase`.
- No cleaning action is required for the `event_type` column.

In [57]:
product_category_counts = (
    df.groupby("product_id")["category_id"]
      .nunique()
)

print("Products linked to more than one category_id:",
      (product_category_counts > 1).sum())

Products linked to more than one category_id: 0


In [59]:
multi_category_products = product_category_counts[
    product_category_counts > 1
].index

df[df["product_id"].isin(multi_category_products)][
    ["product_id", "category_id", "category_code", "brand"]
].drop_duplicates().head(20)

,product_id,category_id,category_code,brand


### Product-to-Category Consistency

- No products were linked to more than one `category_id`.
- Each `product_id` is consistently associated with a single numeric category.

#### Observation

This indicates that the relationship between products and numeric categories is structurally consistent in the October dataset. No cleaning action is required for the `product_id` to `category_id` relationship.

In [62]:
product_brand_counts = (
    df.dropna(subset=["brand"])
      .groupby("product_id")["brand"]
      .nunique()
)

print(
    "Products linked to more than one brand:",
    (product_brand_counts > 1).sum()
)

Products linked to more than one brand: 137


In [ ]:
multi_brand_products = product_brand_counts[
    product_brand_counts > 1
].index

df[df["product_id"].isin(multi_brand_products)][
    ["product_id", "brand", "category_id", "category_code"]
].drop_duplicates().head(20)

### Product-to-Brand Consistency

- **137 products** were associated with more than one non-null brand.
- This indicates a small inconsistency in the relationship between `product_id` and `brand`.

#### Observation

A product would normally be expected to have one consistent brand. Products linked to multiple brands therefore require further investigation.

No cleaning action has been taken yet because the inconsistency could result from:
- changes or corrections in product metadata,
- inconsistent brand labelling,
- data collection issues,
- or different brand values being assigned to the same product over time.

These records will be reviewed before deciding how the brand field should be cleaned.

In [65]:
multi_brand_products = product_brand_counts[
    product_brand_counts > 1
].index

df[df["product_id"].isin(multi_brand_products)][
    ["product_id", "brand", "category_id", "category_code"]
].drop_duplicates().sort_values("product_id").head(30)

,product_id,brand,category_id,category_code
26144151,1306881,lenovo,2053013558920217191,computers.notebook
322358,1306881,dell,2053013558920217191,computers.notebook
39188202,2800241,indesit,2053013563835941749,appliances.kitchen.refrigerators
407768,2800241,leadbros,2053013563835941749,appliances.kitchen.refrigerators
5523,4300224,timberk,2053013552385491165,NaN
34310842,4300224,ibaby,2053013552385491165,NaN
22089104,4502735,NaN,2053013563877884791,appliances.kitchen.hob
22334031,4502735,dauscher,2053013563877884791,appliances.kitchen.hob
22993239,4502735,maunfeld,2053013563877884791,appliances.kitchen.hob
22239558,4502742,indesit,2053013563877884791,appliances.kitchen.hob


### Multi-Brand Product Investigation

Further inspection confirmed that some individual `product_id` values are associated with genuinely different brand names.

Examples include:

- One product associated with both `lenovo` and `dell`.
- One product associated with `indesit` and `leadbros`.
- Several products associated with different tyre brands such as `aeolus` and `roadx`.
- Some products also contain both missing and non-missing brand information.

These differences are not simply caused by capitalisation or minor spelling variations.

#### Data Quality Implication

The `brand` field cannot automatically be assumed to be a stable attribute for every `product_id`.

Possible explanations include:

- product metadata changing over time,
- incorrect brand assignments,
- reused product identifiers,
- or inconsistencies in the source tracking system.

Because the correct brand cannot yet be determined with certainty, no values will be changed during the audit stage. The affected products will require a documented cleaning decision before brand-level analysis is performed.

In [68]:
category_code_counts = (
    df.dropna(subset=["category_code"])
      .groupby("category_id")["category_code"]
      .nunique()
)

print(
    "Category IDs linked to multiple category codes:",
    (category_code_counts > 1).sum()
)

Category IDs linked to multiple category codes: 0


### Category Code Consistency

- No `category_id` values were associated with more than one non-null `category_code`.
- This indicates that the relationship between numeric category IDs and readable category codes is consistent.

#### Observation

The category hierarchy appears structurally reliable. Although `category_code` contains substantial missing data, there is no evidence that the same `category_id` is being assigned conflicting category labels.

This may make it possible to recover some missing `category_code` values from other records sharing the same `category_id`.

In [71]:
category_lookup = (
    df.dropna(subset=["category_code"])
      .drop_duplicates("category_id")
      .set_index("category_id")["category_code"]
)

recoverable_categories = (
    df["category_code"].isna()
    & df["category_id"].isin(category_lookup.index)
)

print(
    "Missing category_code rows potentially recoverable:",
    recoverable_categories.sum()
)

Missing category_code rows potentially recoverable: 0


### Missing Category Code Investigation

- Approximately **31.84%** of records have a missing `category_code`.
- No missing `category_code` records could be recovered using another record with the same `category_id`.
- Category IDs with missing readable category codes consistently remain without a category code throughout the October dataset.

#### Cleaning Implication

The missing category labels cannot be reliably reconstructed from the available October data.

These records should therefore be retained, while the missing category information may later be represented as `Unknown` or another clearly documented reporting category rather than assigning an unsupported product category.

In [74]:
brand_counts = (
    df.dropna(subset=["brand"])
      .groupby("product_id")["brand"]
      .nunique()
)

single_brand_products = brand_counts[
    brand_counts == 1
].index

In [76]:
recoverable_brands = (
    df["brand"].isna()
    & df["product_id"].isin(single_brand_products)
)

print(
    "Missing brand rows potentially recoverable:",
    recoverable_brands.sum()
)

Missing brand rows potentially recoverable: 171756


# STEP 2 — Data Quality Audit

## Objective

The purpose of Day 2 was to identify missing, duplicated, invalid, or inconsistent data in the October 2019 REES46 dataset before making any cleaning decisions.

No records were removed or modified during this stage.

## 1. Missing Values

The dataset is highly complete for the main behavioural fields.

| Column | Missing % |
|---|---:|
| event_time | 0.00% |
| event_type | 0.00% |
| product_id | 0.00% |
| category_id | 0.00% |
| category_code | **31.84%** |
| brand | **14.41%** |
| price | 0.00% |
| user_id | 0.00% |
| user_session | ~0.00% |

### Findings

- `category_code` has the largest completeness issue, with approximately **31.84%** missing.
- `brand` is missing from approximately **14.41%** of records.
- Only **2 `user_session` records** are missing.
- Core event, product, category ID, price, and user fields are otherwise complete.

## 2. Duplicate Audit

- **30,220 additional exact duplicate rows** were identified.
- These represent approximately **0.0712%** of the full dataset.
- When all rows involved in duplicate groups were included using `keep=False`, **48,319 rows** were identified.

Duplicate-group event distribution:

| Event Type | Records |
|---|---:|
| Cart | 44,067 |
| View | 4,100 |
| Purchase | 152 |

### Findings

Most duplicate-looking records are cart events.

Because identical behavioural events may represent repeated customer actions rather than erroneous duplication, these records should **not be automatically removed**.

Duplicate handling requires further investigation during the cleaning stage.

## 3. Price Quality

| Check | Result |
|---|---:|
| Zero-price records | **68,673** |
| Zero-price % | **0.162%** |
| Negative prices | **0** |
| Prices above 2,000 | **227,000** |
| Prices above 2,000 % | **0.535%** |

### Findings

- No negative prices were found.
- Zero-price records require further investigation.
- Prices above 2,000 should not automatically be considered errors because the dataset contains multiple product categories and potentially expensive products.
- High prices require contextual analysis by product/category rather than automatic removal.

## 4. Identifier Validation

- Missing `product_id`: **0**
- Missing `category_id`: **0**
- Missing `user_id`: **0**
- Missing `user_session`: **2**
- Product IDs ≤ 0: **0**
- Category IDs ≤ 0: **0**
- User IDs ≤ 0: **0**

### Findings

The main identifier fields are structurally clean and contain no obvious invalid numeric values.

## 5. Event Type Validation

The dataset contains exactly three event types:

- `view`
- `cart`
- `purchase`

No missing, unexpected, or inconsistent event-type values were identified.

### Cleaning Decision

No cleaning is required for `event_type`.

## 6. Product-to-Category Consistency

- Products linked to multiple `category_id` values: **0**

### Finding

Each `product_id` is consistently associated with one `category_id`.

No structural issue was identified in this relationship.

## 7. Product-to-Brand Consistency

- Products associated with more than one non-null brand: **137**

Examples included products associated with genuinely different brands rather than simple spelling or capitalisation differences.

Examples observed included:

- Lenovo vs Dell
- Indesit vs Leadbros
- Aeolus vs RoadX

### Finding

The `brand` field cannot always be treated as a stable product attribute.

Possible causes include:

- incorrect metadata,
- reused identifiers,
- product metadata changes,
- or source-system inconsistencies.

These products require a documented cleaning decision before brand-level analysis.

## 8. Category-Code Consistency

- Category IDs linked to multiple category codes: **0**

### Finding

Where `category_code` is present, the relationship between `category_id` and `category_code` is consistent.

## 9. Recoverability of Missing Category Codes

- Missing `category_code` values potentially recoverable from another record with the same `category_id`: **0**

### Finding

The missing category labels cannot be reconstructed using other October records.

Therefore, missing category codes should not be assigned inferred categories without supporting evidence.

They may later be retained as missing or represented as a clearly documented `Unknown` category for reporting purposes.

## Main Data Quality Issues Identified

The most important issues to address during cleaning are:

1. **31.84% missing `category_code`**
2. **14.41% missing `brand`**
3. **30,220 exact duplicate rows requiring investigation**
4. **68,673 zero-price records**
5. **137 products assigned to multiple brands**
6. **2 missing `user_session` values**

## STEP 2 Conclusion

The dataset is structurally strong in terms of identifiers, event types, category relationships, and negative-price validation.

The main quality problems are concentrated in **product metadata**, particularly missing categories, missing brands, brand inconsistencies, duplicate-looking behavioural events, and zero-price records.

These issues have now been identified and measured.

No cleaning actions were performed during the audit.

Next stage:

# STEP 3 — Data Cleaning

Day 3 will focus on making justified cleaning decisions for:

- missing `category_code`
- missing `brand`
- recoverable brand values
- multi-brand products
- duplicate-looking rows
- zero-price records
- missing sessions
- and memory-efficient datatypes